<a href="https://colab.research.google.com/github/d-monkey-CG/DFW-Residential-Housing-Price-Predictor/blob/main/DFW_House_PricePredictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This pipeline demonstrates automated model training using AutoGluon, generating a valid regression model for home price prediction. Further optimization can be done with more compute time.


In [ ]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking

In [ ]:
import pandas as pd

from autogluon.tabular import TabularPredictor, TabularDataset

df_train = TabularDataset(data="/DFW_CBINC.csv")

df_test = TabularDataset(data="/DFW_CBINC.csv")

# To create a subsample and set it at approximately 25% of the entire data set
sample_size = int(len(df_train) * 0.25)

print(sample_size)

df_train_small = df_train.sample(sample_size, random_state=42)

Loaded data from: /DFW_CBINC.csv | Columns = 15 / 15 | Rows = 11508395 -> 11508395
Loaded data from: /DFW_CBINC.csv | Columns = 15 / 15 | Rows = 11508395 -> 11508395


2877098


In [ ]:
df_train.head()

,Unnamed: 0,TaxYear,YearBuilt,ActualArea,LandValue,has_pool,IndicatedEquityValue,situsbldgnum,situsstreetprefix,situsstreetsuffix,situsstreetname,situszip,year,county,Median_H_Inc
0,0,2020,2013,6365,171238,True,1235546,6903.0,NaN,DR,AUDUBON,75002,2020.0,Collin,118875.0
1,1,2020,2009,5137,180250,True,936838,6901.0,NaN,DR,AUDUBON,75002,2020.0,Collin,118875.0
2,2,2020,2011,3998,147805,False,544000,6809.0,NaN,DR,AUDUBON,75002,2020.0,Collin,118875.0
3,3,2020,2014,5116,143500,True,757552,6807.0,NaN,DR,AUDUBON,75002,2020.0,Collin,118875.0
4,4,2020,2008,4422,175000,False,805685,6805.0,NaN,DR,AUDUBON,75002,2020.0,Collin,118875.0


In [ ]:
# To define the label

label = "IndicatedEquityValue"
print("Label set to:", label)

Label set to: IndicatedEquityValue


In [ ]:
# Since this is only a proof-of-concept model, I am limiting the model training time to 1 hour

predictor_time = TabularPredictor(label=label, eval_metric='root_mean_squared_error').fit(
    df_train_small,
    time_limit=3600,
    presets="good_quality",
    ag_args_fit={"num_cpus": 1},
    verbosity=2
    )

No path specified. Models will be saved in: "AutogluonModels/ag-20251211_181838"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       5.03 GB / 12.67 GB (39.7%)
Disk Space Avail:   179.93 GB / 225.83 GB (79.7%)
Presets specified: ['good_quality']
Using hyperparameters preset: hyperparameters='light'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting

In [ ]:
# To create a subsample for testing

df_test_small = df_test.sample(sample_size, random_state=42)

# To generate predictions and save them to a csv file

pred_values = predictor_time.predict(df_test_small)
pred_values.head()

df_prediction = pd.DataFrame({
    "ID": df_test_small["Unnamed: 0"].tolist(),
    "EquityValue": pred_values
})

df_prediction.to_csv("prediction.csv", index=False)
df_prediction.head()

,ID,EquityValue
9102981,9102981,415179.500000
6687262,6687262,315466.656250
7035098,7035098,12188.954102
10928356,10928356,368870.750000
3260180,3260180,140741.578125


In [ ]:
# Evaluate model accuracy on the test dataset

results = predictor_time.evaluate(df_test_small)

print("Model Evaluation Metrics:")
for metric, value in results.items():
    print(f"{metric}: {value}")

Model Evaluation Metrics:
root_mean_squared_error: -126946.24865965055
mean_squared_error: -16115346432.0
mean_absolute_error: -37157.28125
r2: 0.8947345614433289
pearsonr: 0.9460857972465703
median_absolute_error: -18605.8125
